In [ ]:
# A ColumnTransformer is a tool (from scikit-learn) that helps to apply different preprocessing steps to different columns of a dataset.
    # In real world data, not all columns are the same:

        # Numerical columns → need scaling/normalization
        # Categorical columns → need encoding
        # Text columns → need vectorization
        # ColumnTransformer handles all of this at once.

# How works
        # Numerical columns → Scaling
        # Categorical columns → Encoding
        # ↓
        # Combine everything into one feature matrix
        # ↓
        # Feed into ML model

In [1]:
# Import required libraries
import pandas as pd
import numpy as np

In [69]:
# Import required classes from skitlearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

In [76]:
# Import required dataset
Data=pd.read_csv('covid_toy.csv')
Data.shape
Data.head(2)

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes


In [ ]:
# Check basic information
Data.info()
# Fever column has null value
# age and fever numerical column
# Other column is categorical

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


In [ ]:
# Check any null value
Data.isnull().sum()

# Fever column contain 10 null value

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [62]:
# check categorical column
Data['cough'].value_counts()
Data['gender'].value_counts()
Data['city'].value_counts()

# Here, cough column is ordinal categorical
# city and gender column are nominal categorical 

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [82]:
# Before transformation split train and test set
X_train, X_test, y_train, y_test=train_test_split(
    Data.drop(columns=['has_covid']),
    Data['has_covid'],
    test_size=0.2
) 
X_train.shape

(80, 5)

In [54]:
# Now perform some feature engineering without column transformer tool

# SimpleImputer -> fever column

# Create simpleimputer object
si=SimpleImputer()

# Now transform the fever column
X_train_fever=si.fit_transform(X_train[['fever']])         # Scikit-learn expects input features as a 2D array
X_test_fever=si.transform(X_test[['fever']])
X_train_fever.shape

(80, 1)

In [58]:
# OrdinalEncoder-> cough column

# Create OrdinalEncoder object
oe=OrdinalEncoder(categories=[['Mild', 'Strong']])

# Fit and transform the cough column
oe.fit(X_train[['cough']])

# Transform the cough column
X_train_cough=oe.transform(X_train[['cough']])
X_test_cough=oe.transform(X_test[['cough']])

In [ ]:
# NominalEncoder-> city and gender column
# create object
oh=OneHotEncoder(drop='first', sparse_output=False)  # Due multiconearity first column drop
# Fit and transform both column
X_train_gender_city=oh.fit_transform(X_train[['gender', 'city']])
X_test_gender_city=oh.transform(X_test[['gender', 'city']])
X_train_gender_city.shape

(80, 4)

In [65]:
# Extract age column and converted pandas dataframe to numpy array
# As preprocessing of encode, simple impute produce numpy array, age coulmn should be also array
X_train_age=X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values  
X_test_age=X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values

In [ ]:
# Now ransformed the column

# Transformed X_train
X_train_tranformed=np.concatenate(
    (X_train_age,
    X_train_fever, 
    X_train_gender_city, 
    X_train_cough),
    axis=1
    )               # axis=1, Joined arrays column wise
# Transformed X_test
X_test_tranformed=np.concatenate(
    (X_test_age,
    X_test_fever, 
    X_test_gender_city, 
    X_test_cough),
    axis=1
    )
X_train_tranformed.shape
X_test_tranformed.shape

(20, 7)

In [83]:
# Now we will do all the process using column_transformer
transformer=ColumnTransformer(
    transformers=[
        ('tnf1', SimpleImputer(), ['fever']),
        ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
        ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
    ], remainder='passthrough'
)

In [84]:
# Now fit and transform the X_train 
X_train=transformer.fit_transform(X_train)


In [ ]:
# Shape of X_train
X_train.shape

(80, 7)

In [86]:
# Now fit and transform the X_test
X_test=transformer.transform(X_test)


In [ ]:
# Shape X_test
X_test.shape

(20, 7)